In [134]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import pickle
import configparser


In [2]:
MiM_path = Path(os.getcwd()).parent
DIR = sys.path.append(MiM_path/'data')
BASE = Path(MiM_path/'data')

# Dos clases (Buena, Mala)

In [231]:
with open(BASE/'labels/HRF.pkl','rb') as f:
    HRF = pickle.load(f)

with open(BASE/'labels/DDR.pkl','rb') as f:
    DDR = pickle.load(f)

with open(BASE/'labels/Kaggle_zhou.pkl','rb') as f:
    zhou = pickle.load(f)

with open(BASE/'labels/DRiD.pkl','rb') as f:
    drid = pickle.load(f)

In [232]:
HRF['dataset'] = 'HRF'
DDR['dataset'] = 'DDR'
zhou['dataset'] = 'Kaggle_Zhou'
drid['dataset'] = 'DRiD'

In [419]:
todos = pd.concat([HRF,DDR,zhou,drid],ignore_index=True)

In [420]:
todos['label'] = todos.label.apply(lambda x: 'Mala' if x == 1 else 'Buena')

In [236]:
global_bin = configparser.ConfigParser()
global_bin.read(BASE/'splits/global_binaria.ini')

['c:\\Users\\marielmoran\\Desktop\\MiM\\data\\splits\\global_binaria.ini']

In [237]:
train = pd.DataFrame({'filename': global_bin['split'].get('training').split(sep=','), 'partition':'train'})
val = pd.DataFrame({'filename': global_bin['split'].get('validation').split(sep=','), 'partition':'val'})
test = pd.DataFrame({'filename': global_bin['split'].get('test').split(sep=','),'partition':'test'})

In [238]:
train['filename'] = train['filename'].str.split(pat='/',expand=True)[1]
val['filename'] = val['filename'].str.split(pat='/',expand=True)[1]
test['filename'] = test['filename'].str.split(pat='/',expand=True)[1]

In [239]:
partition = pd.concat([train,val,test],ignore_index=True)

In [421]:
todos['filename'] = todos['filename'].str.replace('\..*','')

In [422]:
todos = todos.merge(partition, how='left',left_on = 'filename',right_on='filename')

In [423]:
resumen = todos.pivot_table(index='dataset',
                            columns=['partition','label'],
                            aggfunc={'partition':'count'},
                            fill_value= 0)

In [425]:
resumen.columns = resumen.columns.droplevel()

In [426]:
resumen.columns.set_names(['',''],inplace=True)

In [427]:
resumen['test','Total'] = resumen['test','Buena'] + resumen['test','Mala']
resumen['train','Total'] = resumen['train','Buena'] + resumen['train','Mala']
resumen['val','Total'] = resumen['val','Buena'] + resumen['val','Mala']

In [428]:
resumen = resumen.sort_index(axis=1)
resumen = resumen.reindex(level=0,columns=['train','val','test'])

In [429]:
resumen = resumen.append(pd.Series(resumen.sum(axis=0), name='Total'))

In [430]:
resumen['train','Buena'] = round(resumen['train','Buena']/resumen['train','Total'],2).fillna(0)
resumen['train','Mala'] = round(resumen['train','Mala']/resumen['train','Total'],2).fillna(0)

resumen['val','Buena'] = round(resumen['val','Buena']/resumen['val','Total'],2).fillna(0)
resumen['val','Mala'] = round(resumen['val','Mala']/resumen['val','Total'],2).fillna(0)

resumen['test','Buena'] = round(resumen['test','Buena']/resumen['test','Total'],2).fillna(0)
resumen['test','Mala'] = round(resumen['test','Mala']/resumen['test','Total'],2).fillna(0)


In [431]:
resumen

train                val               test             
            Buena  Mala  Total Buena  Mala  Total Buena  Mala  Total
dataset                                                             
DDR          0.92  0.08   6835  0.92  0.08   2733  0.92  0.08   4105
DRiD         0.00  0.00      0  0.00  0.00      0  0.47  0.53   1600
HRF          0.50  0.50     18  0.50  0.50     12  0.50  0.50      6
Kaggle_Zhou  0.96  0.04  35126  0.98  0.02  10906  0.98  0.02  42670
Total        0.96  0.04  41979  0.97  0.03  13651  0.96  0.04  48381

# Tres clases (Buena/Regular/Mala)

In [437]:
with open(BASE/'labels/Retinografia.pkl','rb') as f:
    RETINOGRAFIA = pickle.load(f)

with open(BASE/'labels/Kaggle_fu.pkl','rb') as f:
    FU = pickle.load(f)

In [438]:
RETINOGRAFIA['dataset'] = 'RETINOGRAFIA'
FU['dataset'] = 'FU'

In [439]:
todos3 = pd.concat([RETINOGRAFIA,FU],ignore_index=True)

In [440]:
todos3['label'] = todos3.label.apply(lambda x: 'Mala' if x == 2 else ('Regular' if x == 1 else 'Buena'))

In [441]:
global_tres = configparser.ConfigParser()
global_tres.read(BASE/'splits/global_3cat.ini')

['c:\\Users\\marielmoran\\Desktop\\MiM\\data\\splits\\global_3cat.ini']

In [442]:
train3 = pd.DataFrame({'filename': global_tres['split'].get('training').split(sep=','), 'partition':'train'})
val3 = pd.DataFrame({'filename': global_tres['split'].get('validation').split(sep=','), 'partition':'val'})
test3 = pd.DataFrame({'filename': global_tres['split'].get('test').split(sep=','),'partition':'test'})

In [443]:
train3['filename'] = train3['filename'].str.split(pat='/',expand=True)[1]
val3['filename'] = val3['filename'].str.split(pat='/',expand=True)[1]
test3['filename'] = test3['filename'].str.split(pat='/',expand=True)[1]

In [444]:
partition3 = pd.concat([train3,val3,test3],ignore_index=True)

In [445]:
todos3['filename'] = todos3['filename'].str.replace('\..*','')

In [446]:
todos3 = todos3.merge(partition3, how='left',left_on = 'filename',right_on='filename')

In [447]:
resumen3 = todos3.pivot_table(index='dataset',
                            columns=['partition','label'],
                            aggfunc={'partition':'count'},
                            fill_value= 0)

In [453]:
resumen3.columns = resumen3.columns.droplevel()

In [454]:
resumen3.columns.set_names(['',''],inplace=True)

In [455]:
resumen3['test','Total'] = resumen3['test','Buena'] + resumen3['test','Mala'] + resumen3['test','Regular']
resumen3['train','Total'] = resumen3['train','Buena'] + resumen3['train','Mala'] + resumen3['train','Regular']
resumen3['val','Total'] = resumen3['val','Buena'] + resumen3['val','Mala'] + resumen3['val','Regular']

In [456]:
resumen3 = resumen3.sort_index(axis=1)
resumen3 = resumen3.reindex(level=0,columns=['train','val','test'])

In [457]:
resumen3 = resumen3.append(pd.Series(resumen3.sum(axis=0), name='Total'))

In [460]:
resumen3['train','Buena'] = round(resumen3['train','Buena']/resumen3['train','Total'],2).fillna(0)
resumen3['train','Mala'] = round(resumen3['train','Mala']/resumen3['train','Total'],2).fillna(0)
resumen3['train','Regular'] = round(resumen3['train','Regular']/resumen3['train','Total'],2).fillna(0)

resumen3['val','Buena'] = round(resumen3['val','Buena']/resumen3['val','Total'],2).fillna(0)
resumen3['val','Mala'] = round(resumen3['val','Mala']/resumen3['val','Total'],2).fillna(0)
resumen3['val','Regular'] = round(resumen3['val','Regular']/resumen3['val','Total'],2).fillna(0)

resumen3['test','Buena'] = round(resumen3['test','Buena']/resumen3['test','Total'],2).fillna(0)
resumen3['test','Mala'] = round(resumen3['test','Mala']/resumen3['test','Total'],2).fillna(0)
resumen3['test','Regular'] = round(resumen3['test','Regular']/resumen3['test','Total'],2).fillna(0)


In [462]:
resumen3

train                        val                      test        \
             Buena  Mala Regular  Total Buena  Mala Regular Total Buena  Mala   
dataset                                                                         
FU            0.67  0.18    0.15  12543  0.52  0.20    0.28  8124  0.52  0.20   
RETINOGRAFIA  0.32  0.45    0.23    135  0.30  0.45    0.25    20  0.36  0.43   
Total         0.66  0.19    0.15  12678  0.52  0.20    0.28  8144  0.52  0.20   

                            
             Regular Total  
dataset                     
FU              0.28  8125  
RETINOGRAFIA    0.21    14  
Total           0.28  8139

# Output

In [466]:
resumen.to_excel(BASE/'reports/resumen.xlsx')

In [467]:
resumen3.to_excel(BASE/'reports/resumen3.xlsx')